# Routage des adresses email vers les autorités d'enregistrement

Reprend les familles de signalement issues des classeurs déjà produits par le
chantier email — EGE et PM — et leur affecte une autorité d'enregistrement
à partir de la catégorie d'établissement et du département. Un jeu de fichiers
Excel par autorité est produit séparément pour chaque famille, car elles ne
relèvent pas de la même problématique métier.

Traitement indépendant : ne modifie ni ne redéclenche rien du chantier
email. La catégorie d'établissement est récupérée ici par une requête
directe sur FINESS+, puisqu'elle n'est pas présente dans les exports.

Pour les personnes morales, la catégorie étant un attribut d'établissement,
l'autorité se déduit des fils : héritage direct quand il n'y en a qu'un, famille
majoritaire parmi les domaines des catégories quand il y en a plusieurs.

## 1. Imports et connexion

In [1]:
%run ../../config/config.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src import autorites as aut
from src import routage_contact as rc
from src.chargement import resoudre_tables, charger_categories_ege

pd.set_option("display.max_colwidth", 80)
print("Imports OK")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Connexion OK
Imports OK


## 2. Référentiel des catégories d'établissements

In [2]:
DOSSIER_REF = Path("../../data/referentiels")
CHEMIN_CATEGORIES = DOSSIER_REF / "FINESS_Catégories d'établissements_V3(Ouverts).csv"

referentiel = aut.charger_referentiel(CHEMIN_CATEGORIES)
print(f"{len(referentiel)} catégories chargées")
print(referentiel["statut_referentiel"].value_counts().to_string())
print()
print(referentiel["famille"].value_counts().to_string())

181 catégories chargées
statut_referentiel
RESOLVABLE         176
SANS_AUTORITE        3
MULTI_AUTORITES      2

famille
ARS            135
SOLIDARITES     46


## 3. Catégories des établissements (source live)

In [3]:
TABLES = resoudre_tables(conn, {
    "pm": "PM_SMSSE", "ege": "EGE", "adresse": "Adresse",
    "ege_adresse": "EgeAdresse",
})

categories_ege = charger_categories_ege(conn, TABLES)
print(f"EGE actifs (catégorie + commune) : {len(categories_ege):,}")
print(f"  dont avec catégorie renseignée : "
      f"{categories_ege['TX_CategorieEntiteGeographiqueExercice'].notna().sum():,}")

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   adresse      -> T_FAC_Adresse
   ege_adresse  -> T_FAC_EgeAdresse
EGE actifs (catégorie + commune) : 104,646
  dont avec catégorie renseignée : 104,646


In [4]:
def _colonnes_table(table):
    requete = "SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = ?"
    return set(pd.read_sql(requete, conn, params=[table.split(".")[-1]])["COLUMN_NAME"])

def _choisir(dispo, *candidats):
    for c in candidats:
        if c in dispo:
            return c
    raise KeyError(f"aucune colonne parmi {candidats}")

col_nom_ege = _choisir(_colonnes_table(TABLES["ege"]),
                       "TX_NomEgeLong", "TX_NomEge", "TX_RaisonSocialeEge")
col_nom_pm  = _choisir(_colonnes_table(TABLES["pm"]),
                       "TX_DenominationPm", "TX_RaisonSocialePm", "TX_NomPm")
print(f"   nom EGE -> {col_nom_ege}\n   nom PM  -> {col_nom_pm}")

def _cle(serie):
    return serie.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

identites_ege = pd.read_sql(f"""
    SELECT NB_EgeId,
           TX_NumFinessEge AS numero_finess,
           {col_nom_ege}   AS nom_structure
    FROM {TABLES['ege']} WHERE BL_VersionCourante = 1
""", conn).drop_duplicates("NB_EgeId")

identites_pm = pd.read_sql(f"""
    SELECT NB_PmSmsseId,
           TX_NumFinessPm AS numero_finess,
           {col_nom_pm}   AS nom_structure
    FROM {TABLES['pm']} WHERE BL_VersionCourante = 1
""", conn).drop_duplicates("NB_PmSmsseId")

identites_ege["NB_EgeId"] = _cle(identites_ege["NB_EgeId"])
identites_pm["NB_PmSmsseId"] = _cle(identites_pm["NB_PmSmsseId"])

print(f"\n{len(identites_ege):,} EGE et {len(identites_pm):,} PM identifiés")

   nom EGE -> TX_NomEgeLong
   nom PM  -> TX_DenominationPm

174,644 EGE et 98,217 PM identifiés


## 4. Chargement des classeurs email déjà exportés

In [5]:
RESULTATS = Path("../../results/email")

FAMILLES = [
    ("critiques", "Critiques", ["Anomalies_Critiques"]),
    ("nominatifs", "Nominatifs", ["Nominatifs"]),
    ("indetermines", "Indéterminés", ["Indetermines"]),
    ("vides", "Emails vides", ["Emails_Vides"]),
]

def charger_groupe(fichier, feuilles):
    blocs = [pd.read_excel(fichier, sheet_name=f, dtype=str) for f in feuilles]
    return pd.concat(blocs, ignore_index=True)

sources_ege, sources_pm = {}, {}
for cle, libelle, feuilles in FAMILLES:
    sources_ege[cle] = charger_groupe(RESULTATS / "signalement_emails_ege.xlsx", feuilles)
    sources_pm[cle]  = charger_groupe(RESULTATS / "signalement_emails_pm.xlsx", feuilles)
    print(f"{cle:<14} EGE : {len(sources_ege[cle]):>7,}   PM : {len(sources_pm[cle]):>7,}")

critiques      EGE :     135   PM :      88
nominatifs     EGE :   9,911   PM :   3,762
indetermines   EGE :  24,612   PM :  10,396
vides          EGE :  32,592   PM :  26,552


## 5. Routage par famille

In [6]:
res_critiques_ege = rc.enrichir_libelles(rc.router_ege(sources_ege["critiques"], categories_ege, referentiel))
res_critiques_pm  = rc.enrichir_libelles(rc.router_pm(sources_pm["critiques"], categories_ege, referentiel))

print("EGE :", res_critiques_ege["autorite_statut"].value_counts().to_dict())
print("PM  :", res_critiques_pm["autorite_statut"].value_counts().to_dict())

res_nominatifs_ege = rc.enrichir_libelles(rc.router_ege(sources_ege["nominatifs"], categories_ege, referentiel))
res_nominatifs_pm  = rc.enrichir_libelles(rc.router_pm(sources_pm["nominatifs"], categories_ege, referentiel))

print("EGE :", res_nominatifs_ege["autorite_statut"].value_counts().to_dict())
print("PM  :", res_nominatifs_pm["autorite_statut"].value_counts().to_dict())

res_indetermines_ege = rc.enrichir_libelles(rc.router_ege(sources_ege["indetermines"], categories_ege, referentiel))
res_indetermines_pm  = rc.enrichir_libelles(rc.router_pm(sources_pm["indetermines"], categories_ege, referentiel))

print("EGE :", res_indetermines_ege["autorite_statut"].value_counts().to_dict())
print("PM  :", res_indetermines_pm["autorite_statut"].value_counts().to_dict())

res_vides_ege = rc.enrichir_libelles(rc.router_ege(sources_ege["vides"], categories_ege, referentiel))
res_vides_pm  = rc.enrichir_libelles(rc.router_pm(sources_pm["vides"], categories_ege, referentiel))

print("EGE :", res_vides_ege["autorite_statut"].value_counts().to_dict())
print("PM  :", res_vides_pm["autorite_statut"].value_counts().to_dict())

def enrichir_identite(df, niveau):
    ident = identites_ege if niveau == "ege" else identites_pm
    cle = "NB_EgeId" if niveau == "ege" else "NB_PmSmsseId"
    df = df.copy()
    df[cle] = _cle(df[cle])
    return df.drop(columns=[c for c in ("numero_finess", "nom_structure")
                            if c in df.columns]).merge(ident, on=cle, how="left")

for _famille, _, _ in FAMILLES:
    for _niveau in ("ege", "pm"):
        _nom = f"res_{_famille}_{_niveau}"
        globals()[_nom] = enrichir_identite(globals()[_nom], _niveau)

print("Identités jointes (famille critiques) :",
      f'{res_critiques_ege["numero_finess"].notna().sum()} EGE /',
      f'{res_critiques_pm["numero_finess"].notna().sum()} PM')

EGE : {'RESOLU': 135}
PM  : {'RESOLU': 81, 'NON_DETERMINEE': 5, 'A_ARBITRER': 2}
EGE : {'RESOLU': 9800, 'A_ARBITRER': 104, 'NON_DETERMINEE': 7}
PM  : {'RESOLU': 3464, 'NON_DETERMINEE': 240, 'A_ARBITRER': 58}
EGE : {'RESOLU': 24124, 'A_ARBITRER': 458, 'NON_DETERMINEE': 30}
PM  : {'RESOLU': 9120, 'NON_DETERMINEE': 960, 'A_ARBITRER': 316}
EGE : {'RESOLU': 32325, 'A_ARBITRER': 165, 'NON_DETERMINEE': 102}
PM  : {'RESOLU': 21544, 'NON_DETERMINEE': 4764, 'A_ARBITRER': 244}
Identités jointes (famille critiques) : 135 EGE / 88 PM


## 6. Colonnes à exporter

In [7]:
rc.colonnes_disponibles(*[df for cle, _, _ in FAMILLES
                          for df in (globals()[f"res_{cle}_ege"], globals()[f"res_{cle}_pm"])])

['NB_PmSmsseId',
 'TX_CogCommune',
 'departement',
 'adresse_complete',
 'TEL_MAIL',
 'email_norm',
 'code_anomalie',
 'libelle_anomalie',
 'suggestion',
 'type_local',
 'type_domaine',
 'correspondance',
 'geo_local',
 'geo_concordant',
 'TX_CategorieEntiteGeographiqueExercice',
 'autorite_statut',
 'autorite_code',
 'autorite_candidats',
 'autorite_motif',
 'origine_categorie',
 'autorite_source',
 'type_autorite',
 'region',
 'libelle_region',
 'numero_finess',
 'nom_structure']

In [8]:
COLONNES_EXPORT = [
    "type_structure",
    "autorite_code",
    "libelle_region",
    "departement",

    "cle_structure",
    "TX_CategorieEntiteGeographiqueExercice",
    "numero_finess",
    "nom_structure",
    "TX_CogCommune",
    "adresse_complete",
    "TEL_MAIL",

    "autorite_statut",
    "autorite_source",
    "autorite_candidats",
    "autorite_motif",
    "origine_categorie",
    "nb_ege_fils",
]

# COLONNES_EXPORT = None  # pour tout exporter

## 7. Export par autorité — un jeu de fichiers par famille

In [9]:
DOSSIER_SORTIE = Path("../../results/classement_AE/email")

GROUPES = [
    ("critiques", "Critiques", res_critiques_ege, res_critiques_pm, "Anomalies_Critiques"),
    ("nominatifs", "Nominatifs", res_nominatifs_ege, res_nominatifs_pm, "Nominatifs"),
    ("indetermines", "Indéterminés", res_indetermines_ege, res_indetermines_pm, "Indetermines"),
    ("vides", "Emails vides", res_vides_ege, res_vides_pm, "Emails_Vides"),
]

recaps = {}
for cle, libelle, res_ege, res_pm, feuille in GROUPES:
    combine = rc.combiner(res_ege, res_pm)
    colonnes = [c for c in COLONNES_EXPORT if c in combine.columns] if COLONNES_EXPORT else None
    recap = rc.exporter(combine, DOSSIER_SORTIE / cle, f"EMAIL_{cle.upper()}",
                        cle="cle_structure", nom_feuille=feuille,
                        colonne_repartition="type_structure",
                        colonnes=colonnes)
    recaps[cle] = recap
    print(f"{libelle:<14} : {len(recap)} fichiers (EGE + PM combinés)")

Critiques      : 26 fichiers (EGE + PM combinés)
Nominatifs     : 39 fichiers (EGE + PM combinés)
Indéterminés   : 40 fichiers (EGE + PM combinés)
Emails vides   : 40 fichiers (EGE + PM combinés)


## 8. Contrôle d'exhaustivité

In [10]:
for cle, _, res_ege, res_pm, _ in GROUPES:
    recap = recaps[cle]
    attendu = res_ege["NB_EgeId"].nunique() + res_pm["NB_PmSmsseId"].nunique()
    assert recap["nb_structures"].sum() == attendu, f"écart {cle}"
print("OK — aucune structure perdue ni dupliquée")

OK — aucune structure perdue ni dupliquée
